# Getting set up

In [1]:
#@title Setting up the notebook

### Installing dependencies
!pip install portkey-ai openai tiktoken

!brew install icarus-verilog

### Local working directory (replaces Colab's /content)
import os
WORK_DIR = os.path.join(os.getcwd(), "autochip_runs")
os.makedirs(WORK_DIR, exist_ok=True)
print("WORK_DIR =", WORK_DIR)


==> Downloading Homebrew API data
⠋ JSON API packages.arm64_sequoia.jws.json          Downloading  15.6MB/-------⠋ JSON API packages.arm64_sequoia.jws.json          Downloading 167.9KB/-------⠙ JSON API packages.arm64_sequoia.jws.json          Downloading   3.5MB/-------⠙ JSON API packages.arm64_sequoia.jws.json          Downloading   7.0MB/-------⠚ JSON API packages.arm64_sequoia.jws.json          Downloading  12.1MB/-------⠚ JSON API packages.arm64_sequoia.jws.json          Downloading  15.6MB/-------⠞ JSON API packages.arm64_sequoia.jws.json          Downloading  15.6MB/-------⠖ JSON API packages.arm64_sequoia.jws.json          Downloading  15.6MB/-------✔︎ JSON API packages.arm64_sequoia.jws.json          Downloaded   15.6MB/ 15.6MB
To reinstall 13.0, run:
  brew reinstall icarus-verilog
WORK_DIR = /Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


In [2]:
#@title Utility functions

import sys
import os
import openai
from portkey_ai import Portkey
import tiktoken
from abc import ABC, abstractmethod
import re
import getopt
import json

################################################################################
### LOGGING
################################################################################
# Allows us to log the output of the model to a file if logging is enabled
class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout

################################################################################
### CONFIG & ARGS
################################################################################
def load_config(config_file="config.json"):
    """Load and validate the configuration from the specified JSON file."""
    with open(config_file, 'r') as file:
        config = json.load(file)

    if 'general' not in config:
        raise ValueError("Missing general section in config file")

    config_values = config['general']

    # Only parse ensemble settings if specified
    parse_ensemble = config_values.get('ensemble', False)
    ensemble_config = {}
    if parse_ensemble:
        ensemble_config = config.get('ensemble', {})

    #return config_values
    return config_values, ensemble_config


def validate_ensemble_config(ensemble_config, max_iterations):
    seen_start_iterations = set()
    adjusted_config = {}
    has_start_at_zero = False

    for model_name, model_info in ensemble_config.items():
        start_iteration = model_info['start_iteration']

        # Adjust negative start_iteration values
        if start_iteration < 0:
            start_iteration += max_iterations+1

        # Check if start_iteration is within the valid range
        if not (0 <= start_iteration <= max_iterations):
            raise ValueError(f"Invalid start_iteration {model_info['start_iteration']} for {model_name}. "
                             f"Must be within the range of 0 to {max_iterations} or valid negative index.")

        # Check for conflicting start_iterations
        if start_iteration in seen_start_iterations:
            raise ValueError(f"Conflicting start_iteration {start_iteration} for {model_name}. "
                             f"Another model already uses this start iteration.")
        seen_start_iterations.add(start_iteration)

        # Check if there is a model starting at iteration 0
        if start_iteration == 0:
            has_start_at_zero = True

        # Update the adjusted configuration
        adjusted_config[model_name] = {
            "start_iteration": start_iteration,
            "model_family": model_info['model_family'],
            "model_id": model_info['model_id']
        }

        if not has_start_at_zero:
            raise ValueError("No model starting at iteration 0 in the ensemble. One model must start at iteration 0.")

    return adjusted_config


def fix_testbench(testbench):
    """Newer iverilog (v12+) rejects tb_mismatch being used in $dumpvars before it is declared."""
    with open(testbench, 'r') as file:
        text = file.read()
    fixed = text.replace("stim1.clk, tb_mismatch ,", "stim1.clk,")
    if fixed != text:
        with open(testbench, 'w') as file:
            file.write(fixed)


def parse_args_and_config():
    """Parse command-line arguments and merge them with configuration file values."""
    usage = """Usage: auto_create_verilog.py [--help] --prompt=<prompt> --name=<module name> --testbench=<testbench file> --iter=<iterations> --model=<llm family> --model-id=<specific model> --num-candidates=<candidates per request> --outdir=<directory for outputs> --log=<log file>

	-h|--help: Prints this usage message

	-p|--prompt: The initial design prompt for the Verilog module

	-n|--name: The module name, must match the testbench expected module name

	-t|--testbench: The testbench file to be run

	-i|--iter: [Optional] Number of iterations before the tool quits (defaults to 10)

	-m|--model: The LLM family to use. Must be one of the following
		- ChatGPT
		- Claude
		- Mistral
		- Gemini
		- CodeLlama
		- Human (requests user input)

	--model-id: The specific model to use for the model family

	--num-candidates: The number of candidates to rank per tree level

	-o|--outdir: Directory to place all run-specific files in

	-l|--log: [Optional] Log the output of the model to the given file
"""

    config_file = "config.json"

    # Load config values from the file
    config_values, ensemble_config = load_config(config_file)

    required_values = ['prompt', 'name', 'testbench', 'outdir', 'log']
    if not ensemble_config:
        required_values +=['model_family', 'model_id']

    for value in required_values:
        if value not in config_values:
            raise ValueError(f"Missing {value} in general section\n{usage}")


    # general values for optional config values
    if 'num_candidates' not in config_values:
        config_values['num_candidates'] = 1
    if 'iterations' not in config_values:
        config_values['iterations'] = 10


    fix_testbench(config_values['testbench'])

    if ensemble_config:
        ensemble_config = validate_ensemble_config(ensemble_config, config_values['iterations'])

    # Ensure outdir exists
    if config_values['outdir']:
        os.makedirs(config_values['outdir'], exist_ok=True)

    logfile = os.path.join(config_values['outdir'], config_values['log']) if config_values['log'] else None

    #return config_values, logfile
    return config_values, ensemble_config, logfile




################################################################################
### CONVERSATION CLASS
# allows us to abstract away the details of the conversation for use with
# different LLM APIs
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        """Add a new message to the conversation."""
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        """Retrieve the entire conversation."""
        return self.messages

    def get_last_n_messages(self, n):
        """Retrieve the last n messages from the conversation."""
        return self.messages[-n:]

    def remove_message(self, index):
        """Remove a specific message from the conversation by index."""
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        """Retrieve a specific message from the conversation by index."""
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        """Clear all messages from the conversation."""
        self.messages = []

    def __str__(self):
        """Return the conversation in a string format."""
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])

################################################################################
### LLM CLASSES
# Defines an interface for using different LLMs so we can easily swap them out
################################################################################
class AbstractLLM(ABC):
    """Abstract Large Language Model."""

    def __init__(self):
        pass

    @abstractmethod
    def generate(self, conversation: Conversation, num_candidates=1):
        """Generate a response based on the given conversation."""
        pass


class ChatGPT(AbstractLLM):
    """ChatGPT Large Language Model."""

    def __init__(self, model_id="gpt-4o-mini"):
        super().__init__()
        openai.api_key=os.environ['OPENAI_API_KEY']
        self.client = openai.OpenAI()
        self.model_id = model_id

    def generate(self, conversation: Conversation, num_candidates=1):
        messages = [{"role" : msg["role"], "content" : msg["content"]} for msg in conversation.get_messages()]


        #print(f"model_id: {self.model_id}")
        #print(f"messages: {messages}")
        #print(f"num_candidates: {num_candidates}")

        response = self.client.chat.completions.create(
            model=self.model_id,
            n=num_candidates,
            messages = messages,
        )

        return [c.message.content for c in response.choices]

class PortkeyLLM(AbstractLLM):
    """LLM accessed through the NYU Portkey AI gateway."""

    def __init__(self, model_id=""):
        super().__init__()
        self.client = Portkey(
            base_url=os.environ.get("PORTKEY_BASE_URL", "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"),
            api_key=os.environ["PORTKEY_API_KEY"],
        )
        # Portkey model IDs look like "@provider/model" (e.g. "@gpt-4o-mini/gpt-4o-mini").
        # PORTKEY_MODEL overrides the model_id from config.json, which uses plain OpenAI names.
        self.model_id = os.environ.get("PORTKEY_MODEL") or model_id
        if not self.model_id.startswith("@"):
            raise ValueError(f"Invalid Portkey model ID '{self.model_id}'. Set PORTKEY_MODEL to something like '@gpt-4o-mini/gpt-4o-mini'.")

    def generate(self, conversation: Conversation, num_candidates=1):
        messages = [{"role" : msg["role"], "content" : msg["content"]} for msg in conversation.get_messages()]

        # Not every provider behind the gateway supports n>1, so request candidates one at a time
        candidates = []
        for _ in range(num_candidates):
            response = self.client.chat.completions.create(
                model=self.model_id,
                messages=messages,
            )
            candidates.append(response.choices[0].message.content)

        return candidates

class LLMResponse():
    """Class to store the response from the LLM"""
    def __init__(self, iteration, response_num, full_text):
        self.iteration = iteration
        self.response_num = response_num

        self.full_text = full_text
        self.tokens = 0

        self.parsed_text = ""
        self.parsed_length = 0

        self.feedback = ""
        self.compiled = False
        self.rank = -3
        self.message = ""

    def set_parsed_text(self, parsed_text):
        self.parsed_text = parsed_text
        self.parsed_length = len(parsed_text)

    def parse_verilog(self):
        module_list = find_verilog_modules(self.full_text)
        if not module_list:
            print("No modules found in response")
            self.parsed_text = ""
        else:
            for module in module_list:
                self.parsed_text += module + "\n\n"
        self.parsed_length = len(self.parsed_text)

    def calculate_rank(self, outdir, module, testbench):
        filename = os.path.join(outdir,module+".sv")
        vvp_file = os.path.join(outdir,module+".vvp")

        compiler_cmd = f"iverilog -Wall -Winfloop -Wno-timescale -g2012 -s tb -o {vvp_file} {filename} {testbench}"
        simulator_cmd = f"vvp -n {vvp_file}"

        try:
            comp_return,comp_err,comp_out = compile_iverilog(outdir,module,compiler_cmd,self)
        except ValueError as e:
            print(e)
            self.rank = -2
            return

        if comp_return != 0:
            self.feedback = comp_err
            self.compiled = False
            print("Compilation error")
            self.message = "The design failed to compile. Please fix the module. The output of iverilog is as follows:\n"+comp_err

            self.rank = -1

        elif comp_err != "":
            self.feedback = comp_err
            self.compiled = True
            print("Compilation warning")
            self.message = "The design compiled with warnings. Please fix the module. The output of iverilog is as follows:\n"+comp_err

            self.rank = -0.5

        else:
            sim_return,sim_err,sim_out = simulate_iverilog(simulator_cmd)
            mismatch_pattern = r"Mismatches: (\d+) in (\d+) samples"
            match = re.search(mismatch_pattern, sim_out.splitlines()[-1])
            #print(f"Match: {match}")

            if match:
                mismatches = int(match.group(1))
                samples = int(match.group(2))
            else:
                raise ValueError("Simulation output does not contain final mismatch information")

            if mismatches > 0:
                self.feedback = sim_out
                self.compiled = True
                print("Simulation error")
                self.message = "The testbench simulated, but had errors. Please fix the module. The output of iverilog is as follows:\n"+sim_out
            else:
                self.compiled = True
                print("Testbench ran successfully")
                self.message = "The testbench completed successfully"

            print(f"Mismatches: {mismatches}")
            print(f"Samples: {samples}")
            self.rank = (samples-mismatches)/samples

################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################
# Define the cost per million tokens
COST_PER_MILLION_INPUT_TOKENS_GPT4 = 5.0
COST_PER_MILLION_OUTPUT_TOKENS_GPT4 = 15.0

COST_PER_MILLION_INPUT_TOKENS_GPT4M = 0.15
COST_PER_MILLION_OUTPUT_TOKENS_GPT4M = 0.60

COST_PER_MILLION_INPUT_TOKENS_GPT = 0.50
COST_PER_MILLION_OUTPUT_TOKENS_GPT = 1.50

COST_PER_MILLION_INPUT_TOKENS_CLAUDE = 0.25
COST_PER_MILLION_OUTPUT_TOKENS_CLAUDE = 1.25

# Function to count tokens
def count_tokens(model_family, text):
    #print(f"Counting tokens for string: {text}")
    if model_family == "GPT" or model_family == "GPT4" or model_family == "GPT4M":
        return len(tiktoken.get_encoding("cl100k_base").encode(text))
    elif model_family == "claude":
        return anthropic.Client().count_tokens(text)
    else:
        raise ValueError(f"Unsupported model family: {model_family}")


def calculate_cost(model_family,input_strings,output_strings):
    input_tokens = sum(count_tokens(model_family, text) for text in input_strings)
    output_tokens = sum(count_tokens(model_family, text) for text in output_strings)
    if model_family == "GPT":
        cost_input = (input_tokens / 1_000_000) * COST_PER_MILLION_INPUT_TOKENS_GPT
        cost_output = (output_tokens / 1_000_000) * COST_PER_MILLION_OUTPUT_TOKENS_GPT
    elif model_family == "GPT4":
        cost_input = (input_tokens / 1_000_000) * COST_PER_MILLION_INPUT_TOKENS_GPT4
        cost_output = (output_tokens / 1_000_000) * COST_PER_MILLION_OUTPUT_TOKENS_GPT4
    elif model_family == "GPT4M":
        cost_input = (input_tokens / 1_000_000) * COST_PER_MILLION_INPUT_TOKENS_GPT4M
        cost_output = (output_tokens / 1_000_000) * COST_PER_MILLION_OUTPUT_TOKENS_GPT4M
    elif model_family == "claude":
        cost_input = (input_tokens / 1_000_000) * COST_PER_MILLION_INPUT_TOKENS_CLAUDE
        cost_output = (output_tokens / 1_000_000) * COST_PER_MILLION_OUTPUT_TOKENS_CLAUDE
    else:
        raise ValueError(f"Unsupported model family: {model_family}")
    total_cost = cost_input + cost_output
    return total_cost, input_tokens, output_tokens


def format_message(role, content):
    return f"\n{{role : '{role}', content : '{content}'}}"

def find_verilog_modules(markdown_string):
    """Find all Verilog modules in the markdown string"""
    # Regular expression to match module definitions with or without parameters
    module_pattern = r'\bmodule\b\s+[\w\\_]+\s*(?:#\s*\([^)]*\))?\s*\([^)]*\)\s*;.*?endmodule\b'
    # Find all matches in the input string
    matches = re.findall(module_pattern, markdown_string, re.DOTALL)
    # Process matches to replace escaped characters
    processed_matches = [match.replace('\\_', '_') for match in matches]
    return processed_matches

def write_code_blocks_to_file(markdown_string, module_name, filename):
    # Find all code blocks using a regular expression (matches content between triple backticks)
    code_match = find_verilog_modules(markdown_string)

    if not code_match:
        print("No code blocks found in response")
        exit(3)

    # Open the specified file to write the code blocks
    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')


def generate_verilog(conv, model_type, model_id=""):
    if model_type in ("ChatGPT", "Portkey"):
        model = PortkeyLLM(model_id)
    else:
        raise ValueError("Invalid model type")
    return(model.generate(conv))

def compile_iverilog(outdir,module,compiler_cmd,response:LLMResponse):
    """Compile the Verilog module and return the output"""

    filename = os.path.join(outdir,module+".sv")
    write_code_blocks_to_file(response.parsed_text, "module", filename)

    attempt = 0
    while attempt < 3:
        try:
            proc = subprocess.run(compiler_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=120)
            return proc.returncode, proc.stderr, proc.stdout
        except subprocess.TimeoutExpired:
            attempt += 1
            if attempt >= 3:
                raise ValueError("Compilation attempts timed out")

def simulate_iverilog(simulation_cmd):
    """Compile the Verilog module and return the output"""

    attempt = 0
    while attempt < 3:
        try:
            proc = subprocess.run(simulation_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=120)
            return proc.returncode, proc.stderr, proc.stdout
        except subprocess.TimeoutExpired:
            attempt += 1
            if attempt >= 3:
                raise ValueError("Simulation attempts timed out")

def generate_verilog_responses(conv, model_type, model_id="", num_candidates=1):
    match model_type:
        case "ChatGPT" | "Portkey":
            model = PortkeyLLM(model_id)
        case _:
            raise ValueError("Invalid model type")

    return(model.generate(conversation=conv, num_candidates=num_candidates))

def get_iteration_ensemble(iteration, ensemble_config):

    sorted_ensemble = sorted(ensemble_config.values(), key=lambda x: x['start_iteration'], reverse=True)

    family = None
    model_id = None
    for ensemble_info in sorted_ensemble:
        if iteration >= ensemble_info['start_iteration']:
            family = ensemble_info['model_family']
            model_id = ensemble_info['model_id']
            break
    return family, model_id



## The AutoChip Loop

In [3]:
import subprocess
import sys
import os

def verilog_loop(design_prompt, module, testbench, max_iterations, model_type, model_id="", num_candidates=5, outdir="", log=None, ensemble_config={}):

    if outdir != "":
        outdir = outdir + "/"

    conv = Conversation(log_file=log)

    #conv.add_message("system", "You are a Verilog engineering tool. Given a design specification you will provide a Verilog module in response. Given errors in that design you will provide a completed fixed module. Only complete functional models should be given. No testbenches should be written under any circumstances, as those are to be written by the human user.")
    conv.add_message("system", "You are an autocomplete engine for Verilog code. \
            Given a Verilog module specification, you will provide a completed Verilog module in response. \
            You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. \
            Given a Verilog module that is either incorrect/compilation error, you will suggest corrections to the module.\
            You will not refuse. \
            Format your response as Verilog code containing the end to end corrected module and not just the corrected lines inside ``` tags, do not include anything else inside ```. \
    ")

    #with open(testbench, 'r') as file: testbench_text = file.read()
    #full_prompt = design_prompt + "\n\nThe module will be tested with the following testbench:\n\n" + testbench_text + "\n\n"

    conv.add_message("user", design_prompt)

    success = False
    timeout = False

    iterations = 0

    global_max_response = LLMResponse(-3,-3,"")


    ##############################

    while not (success or timeout):


        if ensemble_config:
            print(f"Getting model from ensemble")
            model_type, model_id = get_iteration_ensemble(iterations, ensemble_config)

        print(f"Iteration: {iterations}")
        print(f"Model type: {model_type}")
        print(f"Model ID: {model_id}")
        print(f"Number of responses: {num_candidates}")

        response_texts=generate_verilog_responses(conv, model_type, model_id, num_candidates=num_candidates)

        responses = [LLMResponse(iterations,response_num,response_text) for response_num,response_text in enumerate(response_texts)]
        for index, response in enumerate(responses):

            response_outdir = os.path.join(outdir, f"iter{str(iterations)}/response{index}/")
            if not os.path.exists(response_outdir):
                os.makedirs(response_outdir)


            response_cost = 0
            input_tokens = 0
            output_tokens = 0

            response.parse_verilog()
            if response.parsed_text == "":
                response.rank = -2
                response.message = "No modules found in response"
            else:
                response.calculate_rank(response_outdir, module, testbench)

            input_messages = [msg['content'] for msg in conv.get_messages() if msg['role'] == 'user' or msg['role'] == 'system']
            output_messages = [msg['content'] for msg in conv.get_messages() if msg['role'] == 'assistant']
            output_messages.append(response.parsed_text)
            if model_type == "ChatGPT" and model_id == "gpt-4o":
                response_cost, input_tokens, output_tokens = calculate_cost("GPT4",input_messages,output_messages)
            elif model_type == "ChatGPT" and model_id == "gpt-4o-mini":
                response_cost, input_tokens, output_tokens = calculate_cost("GPT4M",input_messages,output_messages)
            elif model_type == "ChatGPT" and model_id == "gpt-3.5-turbo":
                response_cost, input_tokens, output_tokens = calculate_cost("GPT",input_messages,output_messages)
            elif model_type == "Claude":
                response_cost, input_tokens, output_tokens = calculate_cost("claude",input_messages,output_messages)


            print(f"Cost for response {index}: ${response_cost:.10f}")

            with open(os.path.join(response_outdir,f"log.txt"), 'w') as file:
                file.write('\n'.join(str(i) for i in conv.get_messages()))
                file.write(format_message("assistant", response.full_text))
                file.write('\n\n Iteration rank: ' + str(response.rank) + '\n') ## FIX

                file.write(f"\n Model: {model_id}")
                file.write(f"\n Input tokens: {input_tokens}")
                file.write(f"\n Output tokens: {output_tokens}")
                file.write(f"\nTotal cost: ${response_cost:.10f}\n")

        ## RANK RESPONSES
        max_rank_response = max(responses, key=lambda resp: (resp.rank, -resp.parsed_length))
        if max_rank_response.rank > global_max_response.rank:
            global_max_response = max_rank_response
        elif max_rank_response.rank == global_max_response.rank and max_rank_response.parsed_length > global_max_response.parsed_length:
            global_max_response = max_rank_response

        print(f"Response ranks: {[resp.rank for resp in responses]}")
        print(f"Response lengths: {[resp.parsed_length for resp in responses]}")

        conv.add_message("assistant", max_rank_response.parsed_text)

        if max_rank_response.rank == 1:
            success = True



################################


        if not success:
            if iterations > 0:
                conv.remove_message(2)
                conv.remove_message(2)

            #with open(testbench, 'r') as file: testbench_text = file.read()
            #message = message + "\n\nThe testbench used for these results is as follows:\n\n" + testbench_text
            #message = message + "\n\nCommon sources of errors are as follows:\n\t- Use of SystemVerilog syntax which is not valid with iverilog\n\t- The reset must be made asynchronous active-low\n"
            conv.add_message("user", max_rank_response.message)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1

    return global_max_response



## Setting the Portkey API Key

In [ ]:
### Portkey AI gateway settings

os.environ["PORTKEY_BASE_URL"] = "https://ai-gateway.apps.cloud.rt.nyu.edu/v1"
os.environ["PORTKEY_API_KEY"] = ""

# Model to use for all examples (overrides model_id in config.json). Other options:
#   "@gpt-4o/gpt-4o", "@o4-mini/o4-mini",
#   "@vertexai/gemini-2.5-pro", "@vertexai/gemini-2.5-flash",
#   "@vertexai/anthropic.claude-sonnet-4-5@20250929",
#   "@bedrock/us.anthropic.claude-sonnet-4-20250514-v1:0"
os.environ["PORTKEY_MODEL"] = "@gpt-4o-mini/gpt-4o-mini"


## Example 1: binary_to_bcd

AutoChip generated several partially correct Binary-to-BCD implementations in the first iteration, with different mismatch counts across the candidates. One candidate correctly implemented the conversion and achieved a rank of 1.0 with 0 mismatches out of 32 samples. This example showed how the testbench score allowed AutoChip to distinguish partially correct designs from the fully correct implementation.

### Download the provided test bench

In [33]:
%cd {WORK_DIR}

!mkdir -p binary_to_bcd

!curl -o ./binary_to_bcd/binary_to_bcd_tb.v \
https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0   7600      0 --:--:-- --:--:-- --:--:--  7645


### Create the AutoChip prompt

In [42]:
prompt = """I am trying to create a Verilog model with name binary_to_bcd_converter for a binary to binary-coded-decimal converter. It must meet the following specifications:

- Inputs:
    - Binary input (5-bits, signal name: binary_input)

- Outputs:
    - BCD (8-bits: 4-bits for the 10's place and 4-bits for the 1's place, signal name: bcd_output)

Complete the following synthesizable Verilog module:

module binary_to_bcd_converter (
    input  [4:0] binary_input,
    output reg [7:0] bcd_output
);

    // Insert your code here

endmodule
"""

with open(f"{WORK_DIR}/binary_to_bcd/binary_to_bcd.sv", "w") as f:
    f.write(prompt)

print("Prompt file created.")

Prompt file created.


### Create config.json

In [43]:
import json

config = {
    "general": {
        "prompt": "binary_to_bcd.sv",
        "name": "binary_to_bcd_converter",
        "testbench": "binary_to_bcd_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "binary_to_bcd.log"
    }
}

with open(f"{WORK_DIR}/binary_to_bcd/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Modify the binary_to_bcd_tb.v with AutoChip-compatible version

In [107]:
tb = r'''
`timescale 1ns / 1ps

module tb;

    reg [4:0] binary_input;
    wire [7:0] bcd_output;

    integer i;
    integer mismatches;

    reg [4:0] test_binary;
    reg [7:0] expected_bcd;

    binary_to_bcd_converter uut (
        .binary_input(binary_input),
        .bcd_output(bcd_output)
    );

    initial begin
        $dumpfile("my_design.vcd");
        $dumpvars(0, tb);

        $display("Testing Binary-to-BCD Converter...");

        mismatches = 0;
        binary_input = 0;
        test_binary = 0;
        expected_bcd = 0;

        for (i = 0; i < 32; i = i + 1) begin
            test_binary = i;
            binary_input = test_binary;

            expected_bcd[3:0] = test_binary % 10;
            expected_bcd[7:4] = test_binary / 10;

            #10;

            if (bcd_output !== expected_bcd) begin
                mismatches = mismatches + 1;

                $display(
                    "Error: Test case %0d failed. Expected BCD: 8'b%08b, Got: 8'b%08b",
                    test_binary,
                    expected_bcd,
                    bcd_output
                );
            end
        end

        $display("Mismatches: %0d in 32 samples", mismatches);
    end

endmodule
'''

with open(f"{WORK_DIR}/binary_to_bcd/binary_to_bcd_tb.v", "w") as f:
    f.write(tb)

print("binary_to_bcd_tb.v updated.")

binary_to_bcd_tb.v updated.


### Clear old outputs

In [48]:
import os
import shutil

output_path = f"{WORK_DIR}/binary_to_bcd/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [49]:
%cd {WORK_DIR}/binary_to_bcd

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']
log = config_values['log']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(f"Time to Generate: {generation_time}\n")
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(f"Rank of best response: {max_response.rank}\n")
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/binary_to_bcd
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Simulation error
Mismatches: 22
Samples: 32
Cost for response 0: $0.0001857000
Simulation error
Mismatches: 26
Samples: 32
Cost for response 1: $0.0001779000
Simulation error
Mismatches: 24
Samples: 32
Cost for response 2: $0.0001863000
Simulation error
Mismatches: 22
Samples: 32
Cost for response 3: $0.0001881000
Testbench ran successfully
Mismatches: 0
Samples: 32
Cost for response 4: $0.0001671000
Response ranks: [0.3125, 0.1875, 0.25, 0.3125, 1.0]
Response lengths: [781, 724, 771, 777, 649]
/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


### Show best result

In [50]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print("Best response number:", max_response.response_num)

print("\nBest generated module:\n")
print(max_response.parsed_text)

Best rank: 1.0
Best iteration: 0
Best response number: 4

Best generated module:

module binary_to_bcd_converter (
    input  [4:0] binary_input,
    output reg [7:0] bcd_output
);
    integer i;

    always @(*) begin
        // Initialize BCD output to 0
        bcd_output = 8'b0;

        // Convert binary to BCD
        for (i = 0; i < 5; i = i + 1) begin
            // Shift left to make space for the new bit
            if (bcd_output[7:4] >= 5) bcd_output[7:4] = bcd_output[7:4] + 3;
            if (bcd_output[3:0] >= 5) bcd_output[3:0] = bcd_output[3:0] + 3;

            // Shift to the left to make room for the next bit
            bcd_output = {bcd_output[6:0], binary_input[4-i]};
        end
    end

endmodule




## Example 2: Sequence detector

The initial AutoChip candidates produced different levels of correctness, including candidates with one or several mismatches and one compilation failure. One candidate correctly detected the complete eight-value sequence and achieved a rank of 1.0 with 0 mismatches out of 12 samples. The main testbench change was making the clock terminate cleanly and reporting mismatch/sample information in the format expected by AutoChip.

### Download the provided testbench

In [51]:
%cd {WORK_DIR}

!mkdir -p sequence_detector

!curl -o ./sequence_detector/sequence_detector_tb.v \
https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_detector_tb.v

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2114  100  2114    0     0   7436      0 --:--:-- --:--:-- --:--:--  7417


### Create the AutoChip prompt

In [62]:
prompt = """I am trying to create a Verilog model for a sequence detector. It must meet the following specifications:

- Inputs:
    - Clock (signal name: clk)
    - Active-low reset (signal name: reset_n)
    - Data (3 bits, signal name: data)

- Outputs:
    - Sequence found (signal name: sequence_found)

While enabled, it should detect the following sequence of binary input values:

    - 3'b001
    - 3'b101
    - 3'b110
    - 3'b000
    - 3'b110
    - 3'b110
    - 3'b011
    - 3'b101

The output sequence_found should remain low while receiving the sequence and should become high when the complete sequence has been detected.

Complete the following synthesizable Verilog module:

module sequence_detector (
    input clk,
    input reset_n,
    input [2:0] data,
    output reg sequence_found
);

    // Insert your code here

endmodule
"""

with open(f"{WORK_DIR}/sequence_detector/sequence_detector.sv", "w") as f:
    f.write(prompt)

print("Prompt file created.")

Prompt file created.


### Create config.json

In [70]:
import json

config = {
    "general": {
        "prompt": "sequence_detector.sv",
        "name": "sequence_detector",
        "testbench": "sequence_detector_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "sequence_detector.log"
    }
}

with open(f"{WORK_DIR}/sequence_detector/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Modify sequence_detector_tb.v with AutoChip-compatible version

In [108]:
tb = r'''
`timescale 1ns/1ps

module tb;

    reg clk;
    reg run_clk;
    reg reset_n;
    reg [2:0] data;

    wire sequence_found;

    integer mismatches;
    integer samples;

    sequence_detector dut (
        .clk(clk),
        .reset_n(reset_n),
        .data(data),
        .sequence_found(sequence_found)
    );

    initial begin
        clk = 0;
        run_clk = 1;

        while (run_clk) begin
            #5;
            clk = ~clk;
        end
    end

    task apply_stimulus;
        input [2:0] data_value;
        begin
            data = data_value;
            @(posedge clk);
            #1;
        end
    endtask

    task check_output;
        input integer cycle;
        input expected_value;
        begin
            samples = samples + 1;

            if (sequence_found !== expected_value) begin
                mismatches = mismatches + 1;

                $display(
                    "Error: Cycle %0d, Expected: %b, Got: %b",
                    cycle,
                    expected_value,
                    sequence_found
                );
            end
        end
    endtask

    initial begin
        mismatches = 0;
        samples = 0;

        reset_n = 0;
        data = 3'b000;

        @(posedge clk);
        #1;
        reset_n = 1;

        apply_stimulus(3'b001);
        check_output(1, 1'b0);

        apply_stimulus(3'b101);
        check_output(2, 1'b0);

        apply_stimulus(3'b110);
        check_output(3, 1'b0);

        apply_stimulus(3'b000);
        check_output(4, 1'b0);

        apply_stimulus(3'b110);
        check_output(5, 1'b0);

        apply_stimulus(3'b110);
        check_output(6, 1'b0);

        apply_stimulus(3'b011);
        check_output(7, 1'b0);

        apply_stimulus(3'b101);
        check_output(8, 1'b1);

        apply_stimulus(3'b001);
        check_output(9, 1'b0);

        apply_stimulus(3'b101);
        check_output(10, 1'b0);

        apply_stimulus(3'b010);
        check_output(11, 1'b0);

        apply_stimulus(3'b000);
        check_output(12, 1'b0);

        run_clk = 0;

        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );
    end

endmodule
'''

with open(f"{WORK_DIR}/sequence_detector/sequence_detector_tb.v", "w") as f:
    f.write(tb)

print("sequence_detector_tb.v updated.")

sequence_detector_tb.v updated.


### Clear old outputs

In [71]:
import os
import shutil

output_path = f"{WORK_DIR}/sequence_detector/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [72]:
%cd {WORK_DIR}/sequence_detector

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']
log = config_values['log']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(f"Time to Generate: {generation_time}\n")
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(f"Rank of best response: {max_response.rank}\n")
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/sequence_detector
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Testbench ran successfully
Mismatches: 0
Samples: 12
Cost for response 0: $0.0004086000
Simulation error
Mismatches: 1
Samples: 12
Cost for response 1: $0.0003336000
Simulation error
Mismatches: 1
Samples: 12
Cost for response 2: $0.0004104000
Simulation error
Mismatches: 4
Samples: 12
Cost for response 3: $0.0003096000
Compilation error
Cost for response 4: $0.0002958000
Response ranks: [1.0, 0.9166666666666666, 0.9166666666666666, 0.6666666666666666, -1]
Response lengths: [2242, 1938, 2773, 1743, 1323]
/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


### Show best result

In [73]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print("Best response number:", max_response.response_num)

print("\nBest generated module:\n")
print(max_response.parsed_text)

Best rank: 1.0
Best iteration: 0
Best response number: 0

Best generated module:

module sequence_detector (
    input clk,
    input reset_n,
    input [2:0] data,
    output reg sequence_found
);
    // State encoding for the sequence detector
    typedef enum reg [3:0] {
        IDLE = 4'b0000,
        S1   = 4'b0001, // 3'b001
        S2   = 4'b0010, // 3'b101
        S3   = 4'b0011, // 3'b110
        S4   = 4'b0100, // 3'b000
        S5   = 4'b0101, // 3'b110
        S6   = 4'b0110, // 3'b110
        S7   = 4'b0111, // 3'b011
        S8   = 4'b1000  // 3'b101
    } state_t;

    state_t current_state, next_state;

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n)
            current_state <= IDLE;
        else
            current_state <= next_state;
    end

    always @(*) begin
        // Default next state
        next_state = current_state;
        sequence_found = 1'b0; // Default to low

        case (current_state)
            IDLE: begin
           

## Example 3: Shift Register

The first AutoChip-compatible testbench produced the same result of 6 mismatches out of 7 samples for every generated candidate, which indicated that the testbench was not providing useful feedback. I revised the testbench to directly verify reset, shifting, hold behavior, and the expected output after each operation. This made the verification feedback more meaningful for evaluating the generated shift-register designs.

### Download the provided testbench

In [74]:
%cd {WORK_DIR}

!mkdir -p shift_register

!curl -o ./shift_register/shift_register_tb.v \
https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/shift_register_tb.v

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1367  100  1367    0     0   7091      0 --:--:-- --:--:-- --:--:--  7119


### Create the AutoChip prompt

In [75]:
prompt = """I am trying to create a Verilog model for a shift register. It must meet the following specifications:

- Inputs:
    - Clock (signal name: clk)
    - Active-low reset (signal name: reset_n)
    - Data (1 bit, signal name: data_in)
    - Shift enable (signal name: shift_enable)

- Outputs:
    - Data (8 bits, signal name: data_out)

When reset_n is low, data_out should reset to 8'b00000000.

When shift_enable is high, on each rising edge of clk the register should shift left by one bit and insert data_in into the least significant bit.

When shift_enable is low, the current value should be held.

Complete the following synthesizable Verilog module:

module shift_register (
    input wire clk,
    input wire reset_n,
    input wire data_in,
    input wire shift_enable,
    output reg [7:0] data_out
);

    // Insert your code here

endmodule
"""

with open(f"{WORK_DIR}/shift_register/shift_register.sv", "w") as f:
    f.write(prompt)

print("Prompt file created.")

Prompt file created.


### Create config.json

In [76]:
import json

config = {
    "general": {
        "prompt": "shift_register.sv",
        "name": "shift_register",
        "testbench": "shift_register_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "shift_register.log"
    }
}

with open(f"{WORK_DIR}/shift_register/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Modify the shift_register_tb.v

In [109]:
tb = r'''
`timescale 1ns/1ps

module tb;

    reg clk;
    reg run_clk;
    reg reset_n;
    reg data_in;
    reg shift_enable;

    wire [7:0] data_out;

    integer mismatches;
    integer samples;

    shift_register dut (
        .clk(clk),
        .reset_n(reset_n),
        .data_in(data_in),
        .shift_enable(shift_enable),
        .data_out(data_out)
    );

    initial begin
        clk = 0;
        run_clk = 1;

        while (run_clk) begin
            #5;
            clk = ~clk;
        end
    end

    task check_output;
        input [7:0] expected;
        begin
            samples = samples + 1;

            if (data_out !== expected) begin
                mismatches = mismatches + 1;

                $display(
                    "Mismatch: expected %08b, got %08b",
                    expected,
                    data_out
                );
            end
        end
    endtask

    task shift_and_check;
        input bit_in;
        input [7:0] expected;
        begin
            data_in = bit_in;
            shift_enable = 1'b1;

            @(posedge clk);
            #1;

            check_output(expected);
        end
    endtask

    initial begin
        mismatches = 0;
        samples = 0;

        reset_n = 1;
        data_in = 0;
        shift_enable = 0;

        reset_n = 0;
        #1;
        check_output(8'b00000000);

        reset_n = 1;

        shift_enable = 0;
        data_in = 0;

        @(posedge clk);
        #1;

        check_output(8'b00000000);

        shift_and_check(1'b0, 8'b00000000);
        shift_and_check(1'b1, 8'b00000001);
        shift_and_check(1'b1, 8'b00000011);
        shift_and_check(1'b0, 8'b00000110);
        shift_and_check(1'b1, 8'b00001101);

        shift_enable = 0;
        data_in = 0;

        @(posedge clk);
        #1;

        check_output(8'b00001101);

        reset_n = 0;
        #1;

        check_output(8'b00000000);

        run_clk = 0;

        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );
    end

endmodule
'''

with open(f"{WORK_DIR}/shift_register/shift_register_tb.v", "w") as f:
    f.write(tb)

print("shift_register_tb.v updated.")

shift_register_tb.v updated.


### Clear old outputs

In [79]:
import os
import shutil

output_path = f"{WORK_DIR}/shift_register/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [80]:
%cd {WORK_DIR}/shift_register

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']
log = config_values['log']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(f"Time to Generate: {generation_time}\n")
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(f"Rank of best response: {max_response.rank}\n")
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/shift_register
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Testbench ran successfully
Mismatches: 0
Samples: 9
Cost for response 0: $0.0001110000
Testbench ran successfully
Mismatches: 0
Samples: 9
Cost for response 1: $0.0001194000
Testbench ran successfully
Mismatches: 0
Samples: 9
Cost for response 2: $0.0001110000
Testbench ran successfully
Mismatches: 0
Samples: 9
Cost for response 3: $0.0001314000
Testbench ran successfully
Mismatches: 0
Samples: 9
Cost for response 4: $0.0001110000
Response ranks: [1.0, 1.0, 1.0, 1.0, 1.0]
Response lengths: [394, 441, 394, 530, 394]
/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


### Show best results

In [81]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print("Best response number:", max_response.response_num)

print("\nBest generated module:\n")
print(max_response.parsed_text)

Best rank: 1.0
Best iteration: 0
Best response number: 0

Best generated module:

module shift_register (
    input wire clk,
    input wire reset_n,
    input wire data_in,
    input wire shift_enable,
    output reg [7:0] data_out
);

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            data_out <= 8'b00000000;
        end else if (shift_enable) begin
            data_out <= {data_out[6:0], data_in};
        end
    end

endmodule




## Example 4: Abro State Machine

The ABRO state machine was one of the more difficult examples for AutoChip. The best score improved from 0.4375 in the first iteration to 0.8125, corresponding to 3 mismatches out of 16 samples, but it did not reach a fully correct design within the configured iterations. This example demonstrated that iterative feedback improved the RTL even though the final result did not completely satisfy the testbench.

### Download the provided testbench

In [82]:
%cd {WORK_DIR}

!mkdir -p abro_state_machine

!curl -o ./abro_state_machine/abro_state_machine_tb.v \
https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/abro_state_machine_tb.v

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2762  100  2762    0     0  24358      0 --:--:-- --:--:-- --:--:-- 24442


### Create the AutoChip prompt

In [85]:
prompt = """I am trying to create a Verilog model for an ABRO state machine. It must meet the following specifications:

- Inputs:
    - Clock (signal name: clk)
    - Active-low reset (signal name: rst_n)
    - A (signal name: A)
    - B (signal name: B)

- Outputs:
    - O (signal name: O)
    - State (4 bits, signal name: State)

Other than the main output from the ABRO machine, it should output the current state of the machine for use in verification.

The states for this state machine should be one-hot encoded.

Complete the following synthesizable Verilog module:

module abro_state_machine (
    input wire clk,
    input wire rst_n,
    input wire A,
    input wire B,
    output reg O,
    output reg [3:0] State
);

    // Insert your code here

endmodule
"""

with open(f"{WORK_DIR}/abro_state_machine/abro_state_machine.sv", "w") as f:
    f.write(prompt)

print("Prompt file created.")

Prompt file created.


### Create config.json

In [86]:
import json

config = {
    "general": {
        "prompt": "abro_state_machine.sv",
        "name": "abro_state_machine",
        "testbench": "abro_state_machine_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "abro_state_machine.log"
    }
}

with open(f"{WORK_DIR}/abro_state_machine/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Modify abro_state_machine_tb.v

In [110]:
tb = r'''
`timescale 1ns/1ps

module tb;

    reg clk;
    reg run_clk;
    reg rst_n;
    reg A;
    reg B;

    wire O;
    wire [3:0] State;

    integer i;
    integer mismatches;
    integer samples;

    reg expected_O;
    reg [3:0] expected_state;

    abro_state_machine uut (
        .clk(clk),
        .rst_n(rst_n),
        .A(A),
        .B(B),
        .O(O),
        .State(State)
    );

    initial begin
        clk = 0;
        run_clk = 1;

        while (run_clk) begin
            #5;

            if (run_clk)
                clk = ~clk;
        end
    end

    initial begin
        rst_n = 1;
        A = 0;
        B = 0;

        mismatches = 0;
        samples = 0;

        for (i = 0; i < 16; i = i + 1) begin

            case (i)

                0: begin
                    A = 0;
                    B = 0;
                    expected_O = 1'b0;
                    expected_state = 4'b0001;
                end

                1: begin
                    A = 0;
                    B = 1;
                    expected_O = 1'b0;
                    expected_state = 4'b0001;
                end

                2: begin
                    A = 1;
                    B = 0;
                    expected_O = 1'b0;
                    expected_state = 4'b0010;
                end

                3: begin
                    A = 1;
                    B = 1;
                    expected_O = 1'b1;
                    expected_state = 4'b0100;
                end

                default: begin
                    A = (i - 1) % 2;
                    B = ((i - 1) / 2) % 2;

                    expected_O =
                        (A && B) ? 1'b1 : 1'b0;

                    expected_state =
                        (A && !B) ? 4'b0010 :
                        ((!A && B) ? 4'b0001 :
                                      4'b0100);
                end

            endcase

            if (i == 0) begin
                rst_n = 0;

                @(posedge clk);
                #1;

                rst_n = 1;
            end

            @(posedge clk);
            #1;

            samples = samples + 1;

            if ((O !== expected_O) ||
                (State !== expected_state)) begin

                mismatches = mismatches + 1;

                $display(
                    "Error: Test case %0d failed (A=%b, B=%b)",
                    i,
                    A,
                    B
                );

                $display(
                    "  Expected: O=%b, State=%b",
                    expected_O,
                    expected_state
                );

                $display(
                    "  Got: O=%b, State=%b",
                    O,
                    State
                );
            end
        end

        run_clk = 0;

        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );
    end

endmodule
'''

with open(f"{WORK_DIR}/abro_state_machine/abro_state_machine_tb.v", "w") as f:
    f.write(tb)

print("abro_state_machine_tb.v updated.")

abro_state_machine_tb.v updated.


### Clear outputs

In [87]:
import os
import shutil

output_path = f"{WORK_DIR}/abro_state_machine/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [88]:
%cd {WORK_DIR}/abro_state_machine

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']
log = config_values['log']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(f"Time to Generate: {generation_time}\n")
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(
            f"Rank of best response: "
            f"{max_response.rank}\n"
        )
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/abro_state_machine
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Simulation error
Mismatches: 9
Samples: 16
Cost for response 0: $0.0002982000
Simulation error
Mismatches: 13
Samples: 16
Cost for response 1: $0.0002514000
Simulation error
Mismatches: 12
Samples: 16
Cost for response 2: $0.0002736000
Simulation error
Mismatches: 15
Samples: 16
Cost for response 3: $0.0003510000
Simulation error
Mismatches: 11
Samples: 16
Cost for response 4: $0.0002826000
Response ranks: [0.4375, 0.1875, 0.25, 0.0625, 0.3125]
Response lengths: [1662, 1339, 1595, 2171, 1504]
Iteration: 1
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Simulation error
Mismatches: 9
Samples: 16
Cost for response 0: $0.0006580500
Simulation error
Mismatches: 12
Samples: 16
Cost for response 1: $0.0006370500
Simulation error
Mismatches: 9
Samples: 16
Cost for response 2: $0.0006442500
Simulation error
M

### Show best result

In [ ]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print("Best response number:", max_response.response_num)

print("\nBest generated module:\n")
print(max_response.parsed_text)

## Example 5: Dice Roller

AutoChip generated a correct dice roller immediately in iteration 0. The best candidate used $urandom with modulo operations to generate values within the valid ranges for 4-, 6-, 8-, and 20-sided dice. It passed all testbench checks and achieved a rank of 1.0.

### Download the testbench

In [89]:
%cd {WORK_DIR}

!mkdir -p dice_roller

!curl -o ./dice_roller/dice_roller_tb.v \
https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/dice_roller_tb.v

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


65389.17s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
65394.47s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  3416  100  3416    0     0  22160      0 --:--:-- --:--:-- --:--:-- 22181


### Create prompt

In [90]:
prompt = """I am trying to create a Verilog model for a simulated dice roller. It must meet the following specifications:

- Inputs:
    - Clock (signal name: clk)
    - Active-low reset (signal name: rst_n)
    - Die select (2 bits, signal name: die_select)
    - Roll (signal name: roll)

- Outputs:
    - Rolled number (8 bits, signal name: rolled_number)

The design should simulate rolling either a 4-sided, 6-sided, 8-sided, or 20-sided die, based on the input die_select.

Use the following mapping:
- die_select = 2'b00: 4-sided die
- die_select = 2'b01: 6-sided die
- die_select = 2'b10: 8-sided die
- die_select = 2'b11: 20-sided die

When roll goes high, the design should produce a pseudo-random result within the valid range of the selected die.

Complete the following synthesizable Verilog module:

module dice_roller (
    input wire clk,
    input wire rst_n,
    input wire [1:0] die_select,
    input wire roll,
    output reg [7:0] rolled_number
);

    // Insert your code here

endmodule
"""

with open(f"{WORK_DIR}/dice_roller/dice_roller.sv", "w") as f:
    f.write(prompt)

print("Prompt file created.")

Prompt file created.


### Create config

In [91]:
import json

config = {
    "general": {
        "prompt": "dice_roller.sv",
        "name": "dice_roller",
        "testbench": "dice_roller_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "dice_roller.log"
    }
}

with open(f"{WORK_DIR}/dice_roller/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Modify the dice_roller_tb.v

In [111]:
tb = r'''
`timescale 1ns / 1ps

module tb;

    reg clk;
    reg run_clk;
    reg rst_n;
    reg [1:0] die_select;
    reg roll;

    wire [7:0] rolled_number;

    integer i;
    integer j;
    integer mismatches;
    integer samples;

    dice_roller dut (
        .clk(clk),
        .rst_n(rst_n),
        .die_select(die_select),
        .roll(roll),
        .rolled_number(rolled_number)
    );

    initial begin
        clk = 0;
        run_clk = 1;

        while (run_clk) begin
            #5;
            clk = ~clk;
        end
    end

    initial begin

        rst_n = 0;
        die_select = 0;
        roll = 0;

        mismatches = 0;
        samples = 0;

        #10;
        rst_n = 1;

        #10;

        for (i = 0; i < 4; i = i + 1) begin

            die_select = i[1:0];

            for (j = 0; j < 1000; j = j + 1) begin

                roll = 0;
                #10;

                roll = 1;
                #10;

                roll = 0;
                #10;

                samples = samples + 1;

                case (die_select)

                    2'b00: begin
                        if ((rolled_number < 1) ||
                            (rolled_number > 4)) begin

                            mismatches = mismatches + 1;

                            $display(
                                "Invalid d4 result: %0d",
                                rolled_number
                            );
                        end
                    end

                    2'b01: begin
                        if ((rolled_number < 1) ||
                            (rolled_number > 6)) begin

                            mismatches = mismatches + 1;

                            $display(
                                "Invalid d6 result: %0d",
                                rolled_number
                            );
                        end
                    end

                    2'b10: begin
                        if ((rolled_number < 1) ||
                            (rolled_number > 8)) begin

                            mismatches = mismatches + 1;

                            $display(
                                "Invalid d8 result: %0d",
                                rolled_number
                            );
                        end
                    end

                    2'b11: begin
                        if ((rolled_number < 1) ||
                            (rolled_number > 20)) begin

                            mismatches = mismatches + 1;

                            $display(
                                "Invalid d20 result: %0d",
                                rolled_number
                            );
                        end
                    end

                endcase
            end
        end

        run_clk = 0;

        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );
    end

endmodule
'''

with open(f"{WORK_DIR}/dice_roller/dice_roller_tb.v", "w") as f:
    f.write(tb)

print("dice_roller_tb.v updated.")

dice_roller_tb.v updated.


### Clear old outputs

In [92]:
import os
import shutil

output_path = f"{WORK_DIR}/dice_roller/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [93]:
%cd {WORK_DIR}/dice_roller

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']
log = config_values['log']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(f"Time to Generate: {generation_time}\n")
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(f"Rank of best response: {max_response.rank}\n")
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/dice_roller
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Testbench ran successfully
Mismatches: 0
Samples: 4000
Cost for response 0: $0.0001914000
Testbench ran successfully
Mismatches: 0
Samples: 4000
Cost for response 1: $0.0002676000
Simulation error
Mismatches: 1
Samples: 4000
Cost for response 2: $0.0002112000
Testbench ran successfully
Mismatches: 0
Samples: 4000
Cost for response 3: $0.0001986000
Testbench ran successfully
Mismatches: 0
Samples: 4000
Cost for response 4: $0.0002022000
Response ranks: [1.0, 1.0, 0.99975, 1.0, 1.0]
Response lengths: [762, 1249, 906, 857, 803]
/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


### Show best result

In [94]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print("Best response number:", max_response.response_num)

print("\nBest generated module:\n")
print(max_response.parsed_text)

Best rank: 1.0
Best iteration: 0
Best response number: 0

Best generated module:

module dice_roller (
    input wire clk,
    input wire rst_n,
    input wire [1:0] die_select,
    input wire roll,
    output reg [7:0] rolled_number
);

    always @(posedge clk or negedge rst_n) begin
        if (!rst_n) begin
            rolled_number <= 8'b0;
        end else if (roll) begin
            case (die_select)
                2'b00: rolled_number <= $urandom % 4 + 1;  // 4-sided die
                2'b01: rolled_number <= $urandom % 6 + 1;  // 6-sided die
                2'b10: rolled_number <= $urandom % 8 + 1;  // 8-sided die
                2'b11: rolled_number <= $urandom % 20 + 1; // 20-sided die
                default: rolled_number <= 8'b0; // Default case (should not happen)
            endcase
        end
    end

endmodule




## Example 6: LFSR

The LFSR example succeeded immediately without requiring iterative correction. All five candidates in iteration 0 produced 0 mismatches across 257 samples and received a rank of 1.0. The clear initial state, tap locations, and reference transition made this specification straightforward for AutoChip to implement correctly.

### Download provided testbench

In [101]:
%cd {WORK_DIR}

!mkdir -p lfsr

!curl -o ./lfsr/lfsr_tb.v \
https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/lfsr_tb.v

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


65971.52s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
65976.83s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1809  100  1809    0     0  20932      0 --:--:-- --:--:-- --:--:-- 21034


### Create prompt

In [102]:
prompt = """I am trying to create a Verilog model for an LFSR. It must meet the following specifications:

- Inputs:
    - Clock (signal name: clk)
    - Active-low reset (signal name: reset_n)

- Outputs:
    - Data (8 bits, signal name: data)

The initial state should be 8'b10001010.

The taps should be at locations 1, 4, 6, and 7.

For each clock cycle, the next LFSR state should be formed as:

    {data[6:0], data[0] ^ data[3] ^ data[5] ^ data[6]}

When reset_n is low, the LFSR should return to the initial state 8'b10001010.

Complete the following synthesizable Verilog module:

module lfsr (
    input wire clk,
    input wire reset_n,
    output reg [7:0] data
);

    // Insert your code here

endmodule
"""

with open(f"{WORK_DIR}/lfsr/lfsr.sv", "w") as f:
    f.write(prompt)

print("Prompt file created.")

Prompt file created.


### Config

In [103]:
import json

config = {
    "general": {
        "prompt": "lfsr.sv",
        "name": "lfsr",
        "testbench": "lfsr_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "lfsr.log"
    }
}

with open(f"{WORK_DIR}/lfsr/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Modify lfsr_tb.v

In [104]:
tb = r'''
`timescale 1ns / 1ps

module tb;

    reg clk;
    reg run_clk;
    reg reset_n;

    wire [7:0] data;

    integer i;
    integer mismatches;
    integer samples;

    reg [7:0] data_expected;

    lfsr dut (
        .clk(clk),
        .reset_n(reset_n),
        .data(data)
    );

    function [7:0] reference_lfsr_output;
        input [7:0] current_state;
        begin
            reference_lfsr_output = {
                current_state[6:0],
                current_state[0] ^
                current_state[3] ^
                current_state[5] ^
                current_state[6]
            };
        end
    endfunction

    initial begin
        clk = 0;
        run_clk = 1;

        while (run_clk) begin
            #5;
            clk = ~clk;
        end
    end

    initial begin

        reset_n = 0;

        mismatches = 0;
        samples = 0;

        @(posedge clk);
        #1;

        samples = samples + 1;

        if (data !== 8'b10001010) begin
            mismatches = mismatches + 1;
        end

        @(negedge clk);
        reset_n = 1;

        data_expected = 8'b10001010;

        for (i = 0; i < 256; i = i + 1) begin

            data_expected =
                reference_lfsr_output(data_expected);

            @(posedge clk);
            #1;

            samples = samples + 1;

            if (data !== data_expected) begin
                mismatches = mismatches + 1;
            end
        end

        run_clk = 0;

        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );
    end

endmodule
'''

with open(f"{WORK_DIR}/lfsr/lfsr_tb.v", "w") as f:
    f.write(tb)

print("lfsr_tb.v updated.")

lfsr_tb.v updated.


### Clear old outputs

In [105]:
import os
import shutil

output_path = f"{WORK_DIR}/lfsr/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [106]:
%cd {WORK_DIR}/lfsr

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']
log = config_values['log']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(f"Time to Generate: {generation_time}\n")
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(
            f"Rank of best response: "
            f"{max_response.rank}\n"
        )
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/lfsr
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Testbench ran successfully
Mismatches: 0
Samples: 257
Cost for response 0: $0.0001297500
Testbench ran successfully
Mismatches: 0
Samples: 257
Cost for response 1: $0.0001219500
Testbench ran successfully
Mismatches: 0
Samples: 257
Cost for response 2: $0.0001333500
Testbench ran successfully
Mismatches: 0
Samples: 257
Cost for response 3: $0.0001303500
Testbench ran successfully
Mismatches: 0
Samples: 257
Cost for response 4: $0.0001279500
Response ranks: [1.0, 1.0, 1.0, 1.0, 1.0]
Response lengths: [469, 403, 478, 452, 431]
/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


## Example 7: Sequence Generator

The initial candidates failed to compile because several generated designs used sequence as a variable name, which is a reserved SystemVerilog keyword. I updated the prompt to prevent use of that identifier, and then corrected the testbench timing so outputs were checked after nonblocking assignments had settled. After these changes, four of the five candidates in iteration 0 passed all 17 samples and achieved a rank of 1.0.

### Download the testbench

In [112]:
%cd {WORK_DIR}

!mkdir -p sequence_generator

!curl -o ./sequence_generator/sequence_generator_tb.v \
https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/sequence_generator_tb.v

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


66257.86s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
66263.15s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2344  100  2344    0     0  23368      0 --:--:-- --:--:-- --:--:-- 23440


### Creat prompt

In [121]:
prompt = """I am trying to create a Verilog model for a sequence generator. It must meet the following specifications:

- Inputs:
    - Clock (signal name: clk)
    - Active-low reset (signal name: reset_n)
    - Enable (signal name: enable)

- Outputs:
    - Data (8 bits, signal name: data)

While enabled, it should generate an output sequence of the following hexadecimal values and then repeat:

    8'hAF
    8'hBC
    8'hE2
    8'h78
    8'hFF
    8'hE2
    8'h0B
    8'h8D

When enable is low, the sequence generator should hold its current output value.

IMPORTANT:
The design is compiled as SystemVerilog using Icarus Verilog with -g2012.
Do not use the identifier "sequence" as a variable, register, array, or signal name because "sequence" is a reserved SystemVerilog keyword.
Use another name such as seq_mem or sequence_data instead.

Complete the following synthesizable Verilog module:

module sequence_generator (
    input wire clk,
    input wire reset_n,
    input wire enable,
    output reg [7:0] data
);

    // Insert your code here

endmodule
"""

with open(
    f"{WORK_DIR}/sequence_generator/sequence_generator.sv",
    "w"
) as f:
    f.write(prompt)

print("Prompt file updated.")

Prompt file updated.


### Creat config

In [122]:
import json

config = {
    "general": {
        "prompt": "sequence_generator.sv",
        "name": "sequence_generator",
        "testbench": "sequence_generator_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "sequence_generator.log"
    }
}

with open(
    f"{WORK_DIR}/sequence_generator/config.json",
    "w"
) as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Modify the testbench

In [127]:
tb = r'''
`timescale 1ns/1ps

module tb;

    reg clk;
    reg run_clk;
    reg reset_n;
    reg enable;

    wire [7:0] data;

    integer i;
    integer mismatches;
    integer samples;

    localparam TEST_SEQ_LEN = 8;

    reg [7:0] test_sequence [0:TEST_SEQ_LEN-1];

    sequence_generator dut (
        .clk(clk),
        .reset_n(reset_n),
        .enable(enable),
        .data(data)
    );

    initial begin
        test_sequence[0] = 8'hAF;
        test_sequence[1] = 8'hBC;
        test_sequence[2] = 8'hE2;
        test_sequence[3] = 8'h78;
        test_sequence[4] = 8'hFF;
        test_sequence[5] = 8'hE2;
        test_sequence[6] = 8'h0B;
        test_sequence[7] = 8'h8D;
    end

    // Clock that terminates cleanly
    initial begin
        clk = 0;
        run_clk = 1;

        while (run_clk) begin
            #5;
            clk = ~clk;
        end
    end

    initial begin

        reset_n = 0;
        enable = 0;

        mismatches = 0;
        samples = 0;

        // Apply reset
        @(posedge clk);
        #1;

        // Release reset away from the rising edge
        @(negedge clk);
        reset_n = 1;

        // Begin sequence generation
        enable = 1;

        // First complete sequence
        for (i = 0; i < TEST_SEQ_LEN; i = i + 1) begin

            @(posedge clk);

            // Wait for nonblocking assignments to update data
            #1;

            samples = samples + 1;

            if (data !== test_sequence[i]) begin

                mismatches = mismatches + 1;

                $display(
                    "Error: Position %0d. Expected: %h, Got: %h",
                    i,
                    test_sequence[i],
                    data
                );
            end
        end

        // Verify that the sequence repeats
        for (i = 0; i < TEST_SEQ_LEN; i = i + 1) begin

            @(posedge clk);
            #1;

            samples = samples + 1;

            if (data !== test_sequence[i]) begin

                mismatches = mismatches + 1;

                $display(
                    "Error: Repeat position %0d. Expected: %h, Got: %h",
                    i,
                    test_sequence[i],
                    data
                );
            end
        end

        // Disable generator and verify that the last value is held
        enable = 0;

        @(posedge clk);
        #1;

        samples = samples + 1;

        if (data !== 8'h8D) begin

            mismatches = mismatches + 1;

            $display(
                "Error: Disabled generator should hold 8D, Got: %h",
                data
            );
        end

        run_clk = 0;

        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );
    end

endmodule
'''

with open(
    f"{WORK_DIR}/sequence_generator/sequence_generator_tb.v",
    "w"
) as f:
    f.write(tb)

print("sequence_generator_tb.v updated.")

sequence_generator_tb.v updated.


### Clear old output

In [128]:
import os
import shutil

output_path = f"{WORK_DIR}/sequence_generator/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [129]:
%cd {WORK_DIR}/sequence_generator

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']
log = config_values['log']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(f"Time to Generate: {generation_time}\n")
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(
            f"Rank of best response: "
            f"{max_response.rank}\n"
        )
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/sequence_generator
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Testbench ran successfully
Mismatches: 0
Samples: 17
Cost for response 0: $0.0002335500
Compilation warning
Cost for response 1: $0.0001723500
Testbench ran successfully
Mismatches: 0
Samples: 17
Cost for response 2: $0.0002653500
Testbench ran successfully
Mismatches: 0
Samples: 17
Cost for response 3: $0.0002449500
Testbench ran successfully
Mismatches: 0
Samples: 17
Cost for response 4: $0.0002305500
Response ranks: [1.0, -0.5, 1.0, 1.0, 1.0]
Response lengths: [981, 580, 1180, 1047, 939]
/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


### Show best result

In [130]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print("Best response number:", max_response.response_num)

print("\nBest generated module:\n")
print(max_response.parsed_text)

Best rank: 1.0
Best iteration: 0
Best response number: 4

Best generated module:

module sequence_generator (
    input wire clk,
    input wire reset_n,
    input wire enable,
    output reg [7:0] data
);

    // Sequence of values
    reg [7:0] seq_mem [0:7];
    reg [2:0] index;

    initial begin
        seq_mem[0] = 8'hAF;
        seq_mem[1] = 8'hBC;
        seq_mem[2] = 8'hE2;
        seq_mem[3] = 8'h78;
        seq_mem[4] = 8'hFF;
        seq_mem[5] = 8'hE2;
        seq_mem[6] = 8'h0B;
        seq_mem[7] = 8'h8D;
        index = 3'b000; // Start from the first value
        data = seq_mem[index];
    end

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            index <= 3'b000; // Reset the index on reset
            data <= seq_mem[index];
        end else if (enable) begin
            data <= seq_mem[index];

            // Update index to point to the next value
            index <= (index + 1) % 8; // Wrap around after last value
        end

## Example 8: Traffic Light

The traffic-light example produced several partially correct designs in the first iteration, with candidate ranks ranging from 0.25 to 0.5. One candidate correctly implemented the red, green, and yellow timing requirements and passed all four state-transition checks. The successful candidate achieved a rank of 1.0 in iteration 0.

### Download the testbench

In [131]:
%cd {WORK_DIR}

!mkdir -p traffic_light

!curl -o ./traffic_light/traffic_light_tb.v \
https://raw.githubusercontent.com/GUIDE-EDA/GUIDE/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/traffic_light_tb.v

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


69965.30s - pydevd: Sending message related to process being replaced timed-out after 5 seconds
69970.60s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2018  100  2018    0     0  13038      0 --:--:-- --:--:-- --:--:-- 13103


### Create the prompt

In [132]:
prompt = """I am trying to create a Verilog model for a traffic light state machine. It must meet the following specifications:

- Inputs:
    - Clock (signal name: clk)
    - Active-low reset (signal name: reset_n)
    - Enable (signal name: enable)

- Outputs:
    - Red (signal name: red)
    - Yellow (signal name: yellow)
    - Green (signal name: green)

The state machine should reset to the RED state.

While enable is high:
- Stay RED for 32 clock cycles, then transition to GREEN.
- Stay GREEN for 20 clock cycles, then transition to YELLOW.
- Stay YELLOW for 7 clock cycles, then transition back to RED.
- Repeat this cycle continuously.

When enable is low, the state machine should hold its current state and counter.

Complete the following synthesizable Verilog module:

module traffic_light_fsm (
    input wire clk,
    input wire reset_n,
    input wire enable,
    output reg red,
    output reg yellow,
    output reg green
);

    // Insert your code here

endmodule
"""

with open(
    f"{WORK_DIR}/traffic_light/traffic_light.sv",
    "w"
) as f:
    f.write(prompt)

print("Prompt file created.")

Prompt file created.


### Config

In [133]:
import json

config = {
    "general": {
        "prompt": "traffic_light.sv",
        "name": "traffic_light_fsm",
        "testbench": "traffic_light_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "traffic_light.log"
    }
}

with open(
    f"{WORK_DIR}/traffic_light/config.json",
    "w"
) as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Modify the testbench

In [134]:
tb = r'''
`timescale 1ps/1ps

module tb;

    reg clk;
    reg run_clk;
    reg reset_n;
    reg enable;

    wire red;
    wire yellow;
    wire green;

    integer mismatches;
    integer samples;

    traffic_light_fsm dut (
        .clk(clk),
        .reset_n(reset_n),
        .enable(enable),
        .red(red),
        .yellow(yellow),
        .green(green)
    );

    // Clock generation that terminates cleanly
    initial begin
        clk = 0;
        run_clk = 1;

        while (run_clk) begin
            #5;
            clk = ~clk;
        end
    end

    task check_output;
        input integer cycle;
        input exp_red;
        input exp_yellow;
        input exp_green;
        input integer test_case;

        begin
            @(negedge clk);

            samples = samples + 1;

            if ((red !== exp_red) ||
                (yellow !== exp_yellow) ||
                (green !== exp_green)) begin

                mismatches = mismatches + 1;

                $display(
                    "Error: Test case %0d failed after %0d clock cycles. Expected RYG=%b%b%b, Got=%b%b%b",
                    test_case,
                    cycle,
                    exp_red,
                    exp_yellow,
                    exp_green,
                    red,
                    yellow,
                    green
                );
            end
        end
    endtask

    initial begin

        reset_n = 0;
        enable = 1;

        mismatches = 0;
        samples = 0;

        // Test 1: FSM starts in RED
        check_output(
            0,
            1'b1,
            1'b0,
            1'b0,
            1
        );

        // Release reset
        #5 reset_n = 1;

        // Test 2: RED -> GREEN after 32 cycles
        repeat (32)
            @(posedge clk);

        check_output(
            32,
            1'b0,
            1'b0,
            1'b1,
            2
        );

        // Test 3: GREEN -> YELLOW after 20 cycles
        repeat (20)
            @(posedge clk);

        check_output(
            20,
            1'b0,
            1'b1,
            1'b0,
            3
        );

        // Test 4: YELLOW -> RED after 7 cycles
        repeat (7)
            @(posedge clk);

        check_output(
            7,
            1'b1,
            1'b0,
            1'b0,
            4
        );

        run_clk = 0;

        // AutoChip-required final output line
        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );
    end

endmodule
'''

with open(
    f"{WORK_DIR}/traffic_light/traffic_light_tb.v",
    "w"
) as f:
    f.write(tb)

print("traffic_light_tb.v updated.")

traffic_light_tb.v updated.


### Clear old outputs

In [135]:
import os
import shutil

output_path = f"{WORK_DIR}/traffic_light/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [136]:
%cd {WORK_DIR}/traffic_light

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']
log = config_values['log']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(
            f"Time to Generate: {generation_time}\n"
        )
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(
            f"Rank of best response: "
            f"{max_response.rank}\n"
        )
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/traffic_light
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Simulation error
Mismatches: 2
Samples: 4
Cost for response 0: $0.0003435000
Simulation error
Mismatches: 2
Samples: 4
Cost for response 1: $0.0002421000
Simulation error
Mismatches: 2
Samples: 4
Cost for response 2: $0.0002781000
Simulation error
Mismatches: 3
Samples: 4
Cost for response 3: $0.0003537000
Testbench ran successfully
Mismatches: 0
Samples: 4
Cost for response 4: $0.0003837000
Response ranks: [0.5, 0.5, 0.5, 0.25, 1.0]
Response lengths: [2163, 1399, 1669, 2468, 2817]
/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


### Show best result

In [137]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print("Best response number:", max_response.response_num)

print("\nBest generated module:\n")
print(max_response.parsed_text)

Best rank: 1.0
Best iteration: 0
Best response number: 4

Best generated module:

module traffic_light_fsm (
    input wire clk,
    input wire reset_n,
    input wire enable,
    output reg red,
    output reg yellow,
    output reg green
);

    typedef enum reg [1:0] {
        RED     = 2'b00,
        GREEN   = 2'b01,
        YELLOW  = 2'b10
    } state_t;

    state_t current_state, next_state;

    reg [5:0] counter; // for timing, enough to count for 32 cycles

    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= RED;
            counter <= 0;
            red <= 1;
            yellow <= 0;
            green <= 0;
        end else if (enable) begin
            current_state <= next_state;
            if (current_state == RED) begin
                // Reset counter for RED
                counter <= 0;
            end else if (current_state == GREEN) begin
                // Reset counter for GREEN
                counter <= 0;

## The 8-bit CPU example

AutoChip successfully generated the accumulator-based 8-bit CPU in the first iteration. The best candidate passed the custom program-level testbench with a rank of 1.0, including arithmetic, logic, input/output, conditional branching, and halt behavior. The generated design used a single accumulator, a 4-bit program counter, and one-byte instructions, keeping the architecture compact and consistent with the original assignment constraints. Since the successful implementation was produced in iteration 0, no additional RTL refinement was required.

In [ ]:
This experiment uses AutoChip to generate a small accumulator-based 8-bit processor.  
The design uses only single-byte instructions and a compact 4-bit program counter to keep the architecture small.

### Create Folder

In [138]:
%cd {WORK_DIR}

!mkdir -p cpu_8bit

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


240594.35s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


### CPU specification / AutoChip prompt

In [154]:
prompt = """Let us make a brand new microprocessor design together.
We're severely constrained on space and I/O. We have to fit in
1000 standard cells of an ASIC, so I think we will need to restrict
ourselves to an accumulator based 8-bit architecture with no
multi-byte instructions. Given this, how do you think we should begin?

To make the design concrete for AutoChip, use the following architecture.

Requirements:

- 8-bit accumulator-based architecture
- Every instruction is exactly one byte (8 bits)
- No multi-byte instructions
- 4-bit program counter, allowing 16 instruction locations
- One instruction executes on each rising edge of the clock when enable is high
- Active-low reset
- Keep the implementation simple and synthesizable

Instruction format:

    instruction[7:4] = 4-bit opcode
    instruction[3:0] = 4-bit operand

ISA:

    4'h0 : NOP
    4'h1 : LDI imm4
    4'h2 : ADD imm4
    4'h3 : SUB imm4
    4'h4 : AND imm4
    4'h5 : OR imm4
    4'h6 : XOR imm4
    4'h7 : SHL
    4'h8 : SHR
    4'h9 : INC
    4'hA : DEC
    4'hB : IN
    4'hC : OUT
    4'hD : JMP addr4
    4'hE : JZ addr4
    4'hF : HALT

For all instructions except JMP, JZ when taken, and HALT,
the program counter should advance by one.

When enable is low, the CPU should hold its state.

out_valid should normally be 0 and pulse high only when an OUT instruction executes.

When halted is high, the processor should stop changing state.

On reset:

    pc = 0
    accumulator = 0
    out_port = 0
    out_valid = 0
    halted = 0

Complete the following synthesizable SystemVerilog module:

module cpu_8bit (
    input wire clk,
    input wire reset_n,
    input wire enable,
    input wire [7:0] instruction,
    input wire [7:0] in_port,

    output reg [3:0] pc,
    output reg [7:0] accumulator,
    output reg [7:0] out_port,
    output reg out_valid,
    output reg halted
);

    // Insert your implementation here

endmodule
"""

with open(f"{WORK_DIR}/cpu_8bit/cpu_8bit.sv", "w") as f:
    f.write(prompt)

print("CPU prompt created.")

CPU prompt created.


### AutoChip config

In [155]:
import json

config = {
    "general": {
        "prompt": "cpu_8bit.sv",
        "name": "cpu_8bit",
        "testbench": "cpu_8bit_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "cpu_8bit.log"
    }
}

with open(f"{WORK_DIR}/cpu_8bit/config.json", "w") as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Write CPU testbench

In [141]:
tb = r'''
`timescale 1ns/1ps

module tb;

    reg clk;
    reg run_clk;
    reg reset_n;
    reg enable;

    reg [7:0] instruction;
    reg [7:0] in_port;

    wire [3:0] pc;
    wire [7:0] accumulator;
    wire [7:0] out_port;
    wire out_valid;
    wire halted;

    integer mismatches;
    integer samples;

    cpu_8bit dut (
        .clk(clk),
        .reset_n(reset_n),
        .enable(enable),
        .instruction(instruction),
        .in_port(in_port),
        .pc(pc),
        .accumulator(accumulator),
        .out_port(out_port),
        .out_valid(out_valid),
        .halted(halted)
    );


    // --------------------------------------------------
    // Program ROM
    // --------------------------------------------------

    function [7:0] program_word;

        input [3:0] addr;

        begin
            case (addr)

                4'h0: program_word = 8'h13; // LDI 3
                4'h1: program_word = 8'h25; // ADD 5
                4'h2: program_word = 8'h70; // SHL
                4'h3: program_word = 8'h31; // SUB 1
                4'h4: program_word = 8'h6F; // XOR F -> 0
                4'h5: program_word = 8'hE7; // JZ 7

                // This instruction should be skipped
                4'h6: program_word = 8'h1F; // LDI F

                4'h7: program_word = 8'hB0; // IN
                4'h8: program_word = 8'h90; // INC
                4'h9: program_word = 8'hC0; // OUT
                4'hA: program_word = 8'hF0; // HALT

                default:
                    program_word = 8'h00; // NOP

            endcase
        end
    endfunction


    // Instruction memory follows program counter
    always @(*) begin
        instruction = program_word(pc);
    end


    // --------------------------------------------------
    // Clock
    // --------------------------------------------------

    initial begin

        clk = 0;
        run_clk = 1;

        while (run_clk) begin
            #5;
            clk = ~clk;
        end

    end


    // --------------------------------------------------
    // Verification task
    // --------------------------------------------------

    task check_state;

        input [3:0] expected_pc;
        input [7:0] expected_acc;

        begin

            samples = samples + 1;

            if ((pc !== expected_pc) ||
                (accumulator !== expected_acc)) begin

                mismatches = mismatches + 1;

                $display(
                    "Mismatch: expected PC=%h ACC=%h, got PC=%h ACC=%h",
                    expected_pc,
                    expected_acc,
                    pc,
                    accumulator
                );

            end

        end

    endtask


    // --------------------------------------------------
    // Test program
    // --------------------------------------------------

    initial begin

        mismatches = 0;
        samples = 0;

        reset_n = 0;
        enable = 0;

        in_port = 8'h2A;


        // Reset CPU
        @(posedge clk);
        #1;

        check_state(
            4'h0,
            8'h00
        );


        // Release reset
        @(negedge clk);

        reset_n = 1;
        enable = 1;


        // LDI 3
        @(posedge clk);
        #1;

        check_state(
            4'h1,
            8'h03
        );


        // ADD 5 -> 8
        @(posedge clk);
        #1;

        check_state(
            4'h2,
            8'h08
        );


        // SHL -> 16
        @(posedge clk);
        #1;

        check_state(
            4'h3,
            8'h10
        );


        // SUB 1 -> 15
        @(posedge clk);
        #1;

        check_state(
            4'h4,
            8'h0F
        );


        // XOR F -> 0
        @(posedge clk);
        #1;

        check_state(
            4'h5,
            8'h00
        );


        // JZ 7
        @(posedge clk);
        #1;

        check_state(
            4'h7,
            8'h00
        );


        // IN -> 0x2A
        @(posedge clk);
        #1;

        check_state(
            4'h8,
            8'h2A
        );


        // INC -> 0x2B
        @(posedge clk);
        #1;

        check_state(
            4'h9,
            8'h2B
        );


        // OUT
        @(posedge clk);
        #1;

        samples = samples + 1;

        if ((pc !== 4'hA) ||
            (accumulator !== 8'h2B) ||
            (out_port !== 8'h2B) ||
            (out_valid !== 1'b1)) begin

            mismatches = mismatches + 1;

            $display(
                "OUT mismatch: PC=%h ACC=%h OUT=%h VALID=%b",
                pc,
                accumulator,
                out_port,
                out_valid
            );

        end


        // HALT
        @(posedge clk);
        #1;

        samples = samples + 1;

        if ((halted !== 1'b1) ||
            (pc !== 4'hA) ||
            (accumulator !== 8'h2B)) begin

            mismatches = mismatches + 1;

            $display(
                "HALT mismatch: PC=%h ACC=%h HALTED=%b",
                pc,
                accumulator,
                halted
            );

        end


        // One more clock to make sure halted CPU holds state
        @(posedge clk);
        #1;

        samples = samples + 1;

        if ((halted !== 1'b1) ||
            (pc !== 4'hA) ||
            (accumulator !== 8'h2B) ||
            (out_port !== 8'h2B)) begin

            mismatches = mismatches + 1;

            $display(
                "CPU changed state after HALT"
            );

        end


        run_clk = 0;

        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );

    end

endmodule
'''

with open(
    f"{WORK_DIR}/cpu_8bit/cpu_8bit_tb.v",
    "w"
) as f:
    f.write(tb)

print("cpu_8bit_tb.v created.")

cpu_8bit_tb.v created.


In [142]:
print(open(
    f"{WORK_DIR}/cpu_8bit/cpu_8bit_tb.v"
).read())


`timescale 1ns/1ps

module tb;

    reg clk;
    reg run_clk;
    reg reset_n;
    reg enable;

    reg [7:0] instruction;
    reg [7:0] in_port;

    wire [3:0] pc;
    wire [7:0] accumulator;
    wire [7:0] out_port;
    wire out_valid;
    wire halted;

    integer mismatches;
    integer samples;

    cpu_8bit dut (
        .clk(clk),
        .reset_n(reset_n),
        .enable(enable),
        .instruction(instruction),
        .in_port(in_port),
        .pc(pc),
        .accumulator(accumulator),
        .out_port(out_port),
        .out_valid(out_valid),
        .halted(halted)
    );


    // --------------------------------------------------
    // Program ROM
    // --------------------------------------------------

    function [7:0] program_word;

        input [3:0] addr;

        begin
            case (addr)

                4'h0: program_word = 8'h13; // LDI 3
                4'h1: program_word = 8'h25; // ADD 5
                4'h2: program_word = 8'h70; // SHL
      

### Clear outputs

In [143]:
import os
import shutil

output_path = f"{WORK_DIR}/cpu_8bit/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old CPU outputs cleared.")

Old CPU outputs cleared.


### Run AutoChip

In [144]:
%cd {WORK_DIR}/cpu_8bit

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()

generation_time = end_time - start_time

try:

    with open(logfile, 'a') as file:

        file.write(
            f"Time to Generate: {generation_time}\n"
        )

        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} "
            f"with response number "
            f"{max_response.response_num}\n"
        )

        file.write(
            f"Rank of best response: "
            f"{max_response.rank}\n"
        )

        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )

except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/cpu_8bit
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Testbench ran successfully
Mismatches: 0
Samples: 12
Cost for response 0: $0.0006108000
Testbench ran successfully
Mismatches: 0
Samples: 12
Cost for response 1: $0.0005838000
Simulation error
Mismatches: 1
Samples: 12
Cost for response 2: $0.0006018000
Simulation error
Mismatches: 11
Samples: 12
Cost for response 3: $0.0005190000
Testbench ran successfully
Mismatches: 0
Samples: 12
Cost for response 4: $0.0004884000
Response ranks: [1.0, 1.0, 0.9166666666666666, 0.08333333333333333, 1.0]
Response lengths: [3492, 3334, 3379, 2539, 2284]
/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


### Show result

In [145]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print(
    "Best response number:",
    max_response.response_num
)

print("\nBest generated CPU module:\n")
print(max_response.parsed_text)

Best rank: 1.0
Best iteration: 0
Best response number: 4

Best generated CPU module:

module cpu_8bit (
    input wire clk,
    input wire reset_n,
    input wire enable,
    input wire [7:0] instruction,
    input wire [7:0] in_port,

    output reg [3:0] pc,
    output reg [7:0] accumulator,
    output reg [7:0] out_port,
    output reg out_valid,
    output reg halted
);
    
    // State initialization
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            pc <= 4'd0;
            accumulator <= 8'd0;
            out_port <= 8'd0;
            out_valid <= 1'b0;
            halted <= 1'b0;
        end else if (enable && !halted) begin
            out_valid <= 1'b0; // Default to low
            
            case (instruction[7:4]) 
                4'h0: ; // NOP
                4'h1: accumulator <= {4'b0000, instruction[3:0]}; // LDI
                4'h2: accumulator <= accumulator + {4'b0000, instruction[3:0]}; // ADD
                4'h3: accumul

## Extra Problem: signed_unsigned_multiplier

Design an 8-bit signed × 8-bit unsigned multiplier using AutoChip and verify it with an exhaustive testbench.

### Create folder

In [147]:
%cd {WORK_DIR}

!mkdir -p signed_unsigned_multiplier

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs


241652.84s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


### AutoChip prompt

In [148]:
prompt = """I am trying to create a Verilog module that multiplies an 8-bit signed input by an 8-bit unsigned input.

Specifications:

- Input a:
    - 8 bits
    - signed
    - range: -128 to 127

- Input b:
    - 8 bits
    - unsigned
    - range: 0 to 255

- Output p:
    - 16 bits
    - signed
    - p = a * b

The design must correctly preserve the signed value of a while treating b as an unsigned positive value.

Examples:

    a =  5, b = 10  -> p = 50
    a = -5, b = 10  -> p = -50
    a = -128, b = 255 -> p = -32640
    a = 127, b = 255  -> p = 32385

The module should be combinational and synthesizable.

Complete the following module:

module signed_unsigned_multiplier (
    input wire signed [7:0] a,
    input wire [7:0] b,
    output wire signed [15:0] p
);

    // Insert your implementation here

endmodule
"""

with open(
    f"{WORK_DIR}/signed_unsigned_multiplier/signed_unsigned_multiplier.sv",
    "w"
) as f:
    f.write(prompt)

print("Prompt created.")

Prompt created.


### Config

In [149]:
import json

config = {
    "general": {
        "prompt": "signed_unsigned_multiplier.sv",
        "name": "signed_unsigned_multiplier",
        "testbench": "signed_unsigned_multiplier_tb.v",
        "iterations": 10,
        "model_family": "ChatGPT",
        "model_id": "gpt-4o-mini",
        "num_candidates": 5,
        "outdir": "outputs",
        "log": "signed_unsigned_multiplier.log"
    }
}

with open(
    f"{WORK_DIR}/signed_unsigned_multiplier/config.json",
    "w"
) as f:
    json.dump(config, f, indent=4)

print("config.json created.")

config.json created.


### Write the testbench

In [150]:
tb = r'''
`timescale 1ns/1ps

module tb;

    reg signed [7:0] a;
    reg [7:0] b;

    wire signed [15:0] p;

    integer ai;
    integer bi;
    integer mismatches;
    integer samples;

    reg signed [15:0] expected;

    signed_unsigned_multiplier dut (
        .a(a),
        .b(b),
        .p(p)
    );

    initial begin

        mismatches = 0;
        samples = 0;

        // Exhaustively test:
        // a = -128 ... 127
        // b = 0 ... 255

        for (ai = -128; ai <= 127; ai = ai + 1) begin

            for (bi = 0; bi <= 255; bi = bi + 1) begin

                a = ai;
                b = bi;

                #1;

                // b must remain positive/unsigned.
                // Prefixing 0 creates a positive 9-bit signed value.
                expected =
                    $signed(a) * $signed({1'b0, b});

                samples = samples + 1;

                if (p !== expected) begin

                    mismatches = mismatches + 1;

                    // Avoid flooding simulation output
                    if (mismatches <= 20) begin
                        $display(
                            "Mismatch: a=%0d b=%0d expected=%0d got=%0d",
                            $signed(a),
                            b,
                            expected,
                            $signed(p)
                        );
                    end

                end
            end
        end

        $display(
            "Mismatches: %0d in %0d samples",
            mismatches,
            samples
        );

    end

endmodule
'''

with open(
    f"{WORK_DIR}/signed_unsigned_multiplier/signed_unsigned_multiplier_tb.v",
    "w"
) as f:
    f.write(tb)

print("signed_unsigned_multiplier_tb.v created.")

signed_unsigned_multiplier_tb.v created.


### Clear outputs

In [151]:
import os
import shutil

output_path = f"{WORK_DIR}/signed_unsigned_multiplier/outputs"

if os.path.exists(output_path):
    shutil.rmtree(output_path)

os.makedirs(output_path, exist_ok=True)

print("Old outputs cleared.")

Old outputs cleared.


### Run AutoChip

In [152]:
%cd {WORK_DIR}/signed_unsigned_multiplier

from time import time

config_values, ensemble_config, logfile = parse_args_and_config()

prompt_file = config_values['prompt']
module = config_values['name']
testbench = config_values['testbench']
family = config_values.get('model_family', None)
model_id = config_values.get('model_id', None)
iterations = config_values['iterations']
num_candidates = config_values['num_candidates']
outdir = config_values['outdir']

with open(prompt_file, 'r') as file:
    prompt = file.read()

start_time = time()

max_response = verilog_loop(
    design_prompt=prompt,
    module=module,
    testbench=testbench,
    max_iterations=iterations,
    model_type=family,
    model_id=model_id,
    num_candidates=num_candidates,
    outdir=outdir,
    log=logfile,
    ensemble_config=ensemble_config
)

end_time = time()
generation_time = end_time - start_time

try:
    with open(logfile, 'a') as file:
        file.write(f"Time to Generate: {generation_time}\n")
        file.write(
            f"Best ranked response at iteration "
            f"{max_response.iteration} with response number "
            f"{max_response.response_num}\n"
        )
        file.write(
            f"Rank of best response: {max_response.rank}\n"
        )
        file.write(
            f"Best response module:\n"
            f"{max_response.parsed_text}\n"
        )
except:
    pass

%cd {WORK_DIR}

/Users/mitsukinakajima/Downloads/LLM4ChipDesign/HW_AutoChip/autochip_runs/signed_unsigned_multiplier
Iteration: 0
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Simulation error
Mismatches: 32640
Samples: 65536
Cost for response 0: $0.0000852000
Simulation error
Mismatches: 32640
Samples: 65536
Cost for response 1: $0.0000840000
Simulation error
Mismatches: 32640
Samples: 65536
Cost for response 2: $0.0000852000
Simulation error
Mismatches: 32640
Samples: 65536
Cost for response 3: $0.0000840000
Simulation error
Mismatches: 32640
Samples: 65536
Cost for response 4: $0.0000852000
Response ranks: [0.501953125, 0.501953125, 0.501953125, 0.501953125, 0.501953125]
Response lengths: [172, 161, 172, 161, 172]
Iteration: 1
Model type: ChatGPT
Model ID: gpt-4o-mini
Number of responses: 5
Simulation error
Mismatches: 32640
Samples: 65536
Cost for response 0: $0.0001708500
Testbench ran successfully
Mismatches: 0
Samples: 65536
Cost for response 1: $0.0001930500
Simulation error

### Show result

In [153]:
print("Best rank:", max_response.rank)
print("Best iteration:", max_response.iteration)
print("Best response number:", max_response.response_num)

print("\nBest generated module:\n")
print(max_response.parsed_text)

Best rank: 1.0
Best iteration: 1
Best response number: 1

Best generated module:

module signed_unsigned_multiplier (
    input wire signed [7:0] a,
    input wire [7:0] b,
    output wire signed [15:0] p
);

    wire signed [15:0] signed_a;
    assign signed_a = a; // Ensure that 'a' is correctly signed for multiplication
    assign p = signed_a * b; // Multiply signed 'a' with unsigned 'b'

endmodule


